**Jacob Petty, Saad Khan, and Sardor Nodirov**

Spring 2026

CS 443: Bio-inspired Machine Learning

# Project 2 | Predictive Coding

This project leverages insights about the brain from neuroscience to build neural networks for image classification and generative AI applications. In the canonical neural networks covered thus far, information flowing *forward* from the input to output layers drives learning. Predictive coding flips this notion completely on its head and **feedback connections** that pass information *backward* from the output layer to earlier layers now play a *major* role. Crucially, feedback allows the *expectation* of what samples of each class *should look like* to shape the learning process.

After building a **predictive coding network (PCN)** using your neural network library and using the net to classify MNIST and CIFAR-10 images, the tasks this week explore two **generative** computational consequences of the extensive feedback:

1. Visualizing the network's expected samples of each image class.
2. "Filling in" images with partially removed (*occluded*) information.

#### Week 1: Building a predictive coding network

The focus this week is to build a densely connected PCN that will introduces us to the concepts and capabilities of predictive coding. Afterwards, we will extend the predictive coding mechanisms to a convolutional neural network, which will allow us to achieve substantially higher image classification accuracy.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.show()
plt.rcParams.update({'font.size': 18})

np.set_printoptions(suppress=True, precision=3)

%load_ext autoreload
%autoreload 2

## Task 0: Copy over neural network library

Copy over the following files from Project 1 to your working project directory: `image_datasets.py`, `network.py`, `layers.py`, `tf_util.py`.

## Task 1: Implement Predictive Coding Network layers

The objective of this task is to implement and test PC layer classes in `dense_pcn_layer.py`.

### 1a. Implement `PCNLayer` class

This is the parent class of all layers in the dense PCN that you are building and contains the methods and variables that all types of dense PCN layers have in common. The `PCNLayer` class itself inherits from your `Dense` class, which allows you to reuse code much of your `Dense` layer code (e.g. weight creation, activation functions) and integrate with the rest of your neural network library. You will extend `Dense` with PCN-specific features, including the **state** of each neuron, which generalizes the concept of `net_act` with **medium-term memory**.

Implement the following methods and use the code below to check your work.
- `PCNLayer` constructor.
- Layer get/set methods (`get_prev_layer`, `get_next_layer`, `set_next_layer`, `is_output_layer`).
- State methods: (`get_state`, `reset_state`, `set_state`, `clamp_state`, `unclamp_state`)
- `__call__`: Do the forward pass through a `PCNLayer`.

In [2]:
from dense_pcn_layer import PCNLayer

#### Test `PCNLayer` layer association

In [3]:
# Test 1: Single PCNLayer
lay1 = PCNLayer('Lay1', units=3, activation='linear', gamma_lr=0.2)
print(lay1.get_name(), ':', lay1.get_num_units(), 'units.' ' gamma=', lay1.gamma_lr)
print('you should see the following above:')
print('Lay1 : 3 units. gamma= 0.2')

Lay1 : 3 units. gamma= 0.2
you should see the following above:
Lay1 : 3 units. gamma= 0.2


I0000 00:00:1773420667.325706 3699055 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1773420667.325968 3699055 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
# Test 1: Multiple PCNLayer
lay1 = PCNLayer('Lay1', units=1, activation='linear')
lay2 = PCNLayer('Lay2', units=10, activation='linear', prev_layer_or_block=lay1)
lay3 = PCNLayer('Lay3', units=20, activation='linear', prev_layer_or_block=lay2)
lay4 = PCNLayer('Lay4', units=30, activation='linear', prev_layer_or_block=lay3)

lay1.set_next_layer(lay2)
lay2.set_next_layer(lay3)
lay3.set_next_layer(lay4)

layers = [lay1, lay2, lay3, lay4]

for lay in layers:
    print(lay.get_name(), ':', lay.get_num_units(), end=' ')
print('You should see:')
print('Lay1 : 1 Lay2 : 10 Lay3 : 20 Lay4 : 30 ')

print('Prev layer test:')
for lay in layers:
    prev_name = lay.get_prev_layer().get_name() if lay.get_prev_layer() is not None else '[No prev]'
    print(lay.get_name(), ' prev layer is:', prev_name)
print('You should see:')
print('''Lay1  prev layer is: [No prev]
Lay2  prev layer is: Lay1
Lay3  prev layer is: Lay2
Lay4  prev layer is: Lay3''')

print('Next layer test:')
for lay in layers:
    next_name = lay.get_next_layer().get_name() if lay.get_next_layer() is not None else '[No next]'
    print(lay.get_name(), ' next layer is:', next_name)
print('You should see:')
print('''Lay1  next layer is: Lay2
Lay2  next layer is: Lay3
Lay3  next layer is: Lay4
Lay4  next layer is: [No next]''')

print('The next line should print out False 4 times:')
for lay in layers:
    print(lay.is_output_layer(), end=' ')

Lay1 : 1 Lay2 : 10 Lay3 : 20 Lay4 : 30 You should see:
Lay1 : 1 Lay2 : 10 Lay3 : 20 Lay4 : 30 
Prev layer test:
Lay1  prev layer is: [No prev]
Lay2  prev layer is: Lay1
Lay3  prev layer is: Lay2
Lay4  prev layer is: Lay3
You should see:
Lay1  prev layer is: [No prev]
Lay2  prev layer is: Lay1
Lay3  prev layer is: Lay2
Lay4  prev layer is: Lay3
Next layer test:
Lay1  next layer is: Lay2
Lay2  next layer is: Lay3
Lay3  next layer is: Lay4
Lay4  next layer is: [No next]
You should see:
Lay1  next layer is: Lay2
Lay2  next layer is: Lay3
Lay3  next layer is: Lay4
Lay4  next layer is: [No next]
The next line should print out False 4 times:
False False False False 

#### Test `PCNLayer` state management 

In [5]:
lay = PCNLayer('TestLayer', units=5, activation='linear')

print(f'Is state initialized to None as it should be? {lay.get_state() is None}')
print('Resetting state...')
lay.reset_state(3)
print(f'Is state initialized to None as it should NOT be? {lay.get_state() is not None}')
print('Current state is:')
print(lay.get_state(), 'and should be')
print('''tf.Tensor(
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]], shape=(3, 5), dtype=float32)''')
lay.set_state(tf.reshape(tf.range(15, dtype=tf.float32), [3, 5]))
print('Current state is:')
print(lay.get_state(), 'and should be')
print('''tf.Tensor(
[[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]], shape=(3, 5), dtype=float32)''')
print('Resetting state...')
lay.reset_state(3)
print(f'Reset state is correctly shaped? {tf.reduce_prod(lay.get_state().shape) == 15}')
print(f'Reset state is correct dtype? {lay.get_state().dtype is tf.float32}')
print(f'Current state is correctly reset? {tf.reduce_all(lay.get_state() == 0.)}')

Is state initialized to None as it should be? True
Resetting state...
Is state initialized to None as it should NOT be? True
Current state is:
tf.Tensor(
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]], shape=(3, 5), dtype=float32) and should be
tf.Tensor(
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]], shape=(3, 5), dtype=float32)
Current state is:
tf.Tensor(
[[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]], shape=(3, 5), dtype=float32) and should be
tf.Tensor(
[[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]], shape=(3, 5), dtype=float32)
Resetting state...
Reset state is correctly shaped? True
Reset state is correct dtype? True
Current state is correctly reset? True


In [6]:
lay = PCNLayer('TestLayer', units=4, activation='linear')
print('State is clamped? ', lay.is_clamped.numpy(), '. It should be False.', sep='')
print('State is tf Variable? ', isinstance(lay.is_clamped, tf.Variable), '. It should be True.', sep='')
print('State is trainable? ', lay.is_clamped.trainable, '. It should be False.', sep='')
print('Clamping layer...')
lay.clamp_state()
print('State is clamped? ', lay.is_clamped.numpy(), '. It should be True.', sep='')
print('State is trainable? ', lay.is_clamped.trainable, '. It should be False.', sep='')
print('Unclamping layer...')
lay.unclamp_state()
print('State is clamped? ', lay.is_clamped.numpy(), '. It should be False.', sep='')
print('State is trainable? ', lay.is_clamped.trainable, '. It should be False.', sep='')

State is clamped? False. It should be False.
State is tf Variable? True. It should be True.
State is trainable? False. It should be False.
Clamping layer...
State is clamped? True. It should be True.
State is trainable? False. It should be False.
Unclamping layer...
State is clamped? False. It should be False.
State is trainable? False. It should be False.


#### Test forward pass

In [7]:
tf.random.set_seed(0)
lay = PCNLayer('TestLayer', units=4, activation='linear')
tf.random.set_seed(1)
test_x = tf.random.uniform(shape=(2, 3), maxval=1.0)
test_net_act = lay(test_x)
print('Test net_act is:')
print(test_net_act.numpy(), 'and should be:')
print('''[[-0.001 -0.006 -0.01  -0.018]
 [-0.004 -0.006 -0.006 -0.008]]''')
print(f'Output shape is {lay.output_shape} and should be [2, 4]')
print('Layer state is:')
print(lay.get_state().numpy(), 'and should be:')
print('''[[-0.001 -0.006 -0.01  -0.018]
 [-0.004 -0.006 -0.006 -0.008]]''')


Test net_act is:
[[-0.001 -0.006 -0.01  -0.018]
 [-0.004 -0.006 -0.006 -0.008]] and should be:
[[-0.001 -0.006 -0.01  -0.018]
 [-0.004 -0.006 -0.006 -0.008]]
Output shape is [2, 4] and should be [2, 4]
Layer state is:
[[-0.001 -0.006 -0.01  -0.018]
 [-0.004 -0.006 -0.006 -0.008]] and should be:
[[-0.001 -0.006 -0.01  -0.018]
 [-0.004 -0.006 -0.006 -0.008]]


### 1b. Implement and test `InputPCNLayer`

Like artificial neural networks, the input layer represents the currently mini-batch of data features. However, the PCN input layer also has a state that behaves differently, depending on whether it is **clamped** to the current mini-batch of images:

- **If clamped:**  The "state" represents the mini-batch of data and it is frozen. This means the mini-batch of data DOES NOT develop/change based on the network activations (i.e. network does not "dream" onto the input).
- **If not clamped:** We allow the feedback activations from the first hidden layer of the PCN to modify the state in the input layer. This will eventually allow us to visualize what *expected* inputs of each class *should look like*, according to the trained PCN (i.e. network "dreams" onto the input).

Because the input layer maintains this state, we create a distinct layer class for the input layer (`InputPCNLayer`) that inherits from your `PCNLayer` parent class to manage state.

Implement and test the following `InputPCNLayer` methods:
- constructor
- `has_wts`: The input layer has no weights (it is the mini-batch + state).
- `compute_net_input`: returns the mini-batch.
- `update_state`: Modifies the state based on feedback from the PCN when the layer is not clamped.

Recall that the input layer's state is updated according to gradient descent:

$$
\text{Mem}_{i, j} (t) = \text{Mem}_{i, j} (t-1) - \gamma \varepsilon_{i,j}
$$

where $\text{Mem}_{i, j} (t)$ is the medium term memory state of input layer neuron $j$ for sample $i$ at time step $t$, $\gamma$ is the feedback signal strength, $\varepsilon_{i,j}$ is the prediction error based on the **top-down feedback** from the next layer alone.

In [8]:
from dense_pcn_layer import InputPCNLayer

#### Test `InputPCNLayer`

We defer testing `update_state` until we have PCN hidden layers.

In [9]:
input_lay = InputPCNLayer('InputLayer', units=8)
print('State is clamped? ', input_lay.is_clamped.numpy(), '. It should be True.', sep='')
print('State is tf Variable? ', isinstance(input_lay.is_clamped, tf.Variable), '. It should be True.', sep='')
print('State is trainable? ', input_lay.is_clamped.trainable, '. It should be False.', sep='')
print(f'Does the layer not have weights like it should not? {not input_lay.has_wts()}')
test_x = 2.0*tf.ones([1, 8])
x_batch = input_lay(test_x)
print('The layer net_act is:')
print(x_batch.numpy(), 'and it should be: [[2. 2. 2. 2. 2. 2. 2. 2.]]')


State is clamped? True. It should be True.
State is tf Variable? True. It should be True.
State is trainable? False. It should be False.
Does the layer not have weights like it should not? True
The layer net_act is:
[[2. 2. 2. 2. 2. 2. 2. 2.]] and it should be: [[2. 2. 2. 2. 2. 2. 2. 2.]]


### 1c. Implement and test `DensePCNLayer`

This represents the interior, dense hidden layers within the the net. It is similar to a regular `Dense` layer except:
- There is no bias.
- Every neuron maintains a state.
- The layer can use the state to predict the expected input in the layer below.

Implement the following in the `DensePCNLayer` class:
- constructor.
- `init_params`: We disable the bias.
- `compute_net_input`: Same as `Dense` except removal of bias.
- `predict_input`: Use the state to predict the layer's expected input.
- `prediction_error`: Compute the prediction error — the difference between the layer below's state and the current layer's predicted input.
- `update_state:` Update the layer's state based on its prediction error, as well as that of the layer above.

Recall that the interior hidden layer's state is updated according to gradient descent:

$$
\text{Mem}_{i, j} (t) = \text{Mem}_{i, j} (t-1) - \gamma \left (\text{d}_{\text{BU}} + \text{d}_{\text{TD}} \right)
$$

where $\text{Mem}_{i, j} (t)$ is the medium term memory state of hidden layer neuron $j$ for sample $i$ at time step $t$ and $\gamma$ is the feedback signal strength.
- $\text{d}_{\text{BU}}$ is the "bottom-up gradient" (`shape: (B, H)`), which is the negated current layer prediction error "forwarded" through the current layer.
- $\text{d}_{\text{TD}}$ is the "top-down gradient" (`shape: (B, H)`), which is the difference between the current layer's state and the next layer's predicted input.

In [10]:
from dense_pcn_layer import DensePCNLayer

#### Test single `DensePCNLayer`

In [11]:
tf.random.set_seed(0)
dense = DensePCNLayer('HiddenLayer', units=5)
dense.init_params((2, 3))
tf.random.set_seed(1)
test_x = tf.random.normal(shape=(2, 3))
test_net_act = dense(test_x)
print('Your net_act is:')
print(test_net_act.numpy(), 'and it should be:')
print('''[[-0.012 -0.007  0.026  0.018  0.019]
 [-0.012  0.001 -0.019  0.009  0.013]] ''')
print('Your state is:')
print(dense.get_state().numpy(), 'and it should be:')
print('''[[-0.012 -0.007  0.026  0.018  0.019]
 [-0.012  0.001 -0.019  0.009  0.013]] ''')

dense.set_state(test_net_act + 1.5)
print('The predicted input of the layer is:')
print(dense.predict_input().numpy(), 'and it should be:')
print('''[[-0.012  0.043 -0.03 ]
 [-0.012  0.043 -0.03 ]]''')


Your net_act is:
[[-0.012 -0.007  0.026  0.018  0.019]
 [-0.012  0.001 -0.019  0.009  0.013]] and it should be:
[[-0.012 -0.007  0.026  0.018  0.019]
 [-0.012  0.001 -0.019  0.009  0.013]] 
Your state is:
[[-0.012 -0.007  0.026  0.018  0.019]
 [-0.012  0.001 -0.019  0.009  0.013]] and it should be:
[[-0.012 -0.007  0.026  0.018  0.019]
 [-0.012  0.001 -0.019  0.009  0.013]] 
The predicted input of the layer is:
[[-0.012  0.043 -0.03 ]
 [-0.012  0.043 -0.03 ]] and it should be:
[[-0.012  0.043 -0.03 ]
 [-0.012  0.043 -0.03 ]]


#### Test prediction error

With input layer and one hidden layer.

In [12]:
tf.random.set_seed(0)
inputlayer = InputPCNLayer('InputLayer', units=6)
tf.random.set_seed(1)
dense = DensePCNLayer('HiddenLayer', units=4, prev_layer_or_block=inputlayer)
inputlayer.set_next_layer(dense)
tf.random.set_seed(2)
test_x = tf.random.normal(shape=(3, 6))
test_net_act = dense(inputlayer(test_x))
print('Your net_act is:')
print(test_net_act.numpy(), 'and it should be:')
print('''[[-0.032 -0.041 -0.056 -0.021]
 [ 0.004  0.012  0.006  0.003]
 [-0.003  0.023 -0.018  0.004]]''')
print('Hidden layer prediction error is:')
print(dense.prediction_error().numpy(), 'and it should be')
print('''[[ 0.436 -1.907  1.378 -1.04  -0.     0.17 ]
 [ 0.916  0.551 -0.135 -0.062 -0.363  0.023]
 [ 0.247 -0.5   -1.299 -0.446  0.461  0.803]]''')

Your net_act is:
[[-0.032 -0.041 -0.056 -0.021]
 [ 0.004  0.012  0.006  0.003]
 [-0.003  0.023 -0.018  0.004]] and it should be:
[[-0.032 -0.041 -0.056 -0.021]
 [ 0.004  0.012  0.006  0.003]
 [-0.003  0.023 -0.018  0.004]]
Hidden layer prediction error is:
[[ 0.436 -1.907  1.378 -1.04  -0.     0.17 ]
 [ 0.916  0.551 -0.135 -0.062 -0.363  0.023]
 [ 0.247 -0.5   -1.299 -0.446  0.461  0.803]] and it should be
[[ 0.436 -1.907  1.378 -1.04  -0.     0.17 ]
 [ 0.916  0.551 -0.135 -0.062 -0.363  0.023]
 [ 0.247 -0.5   -1.299 -0.446  0.461  0.803]]


#### Test `update_state`

With input layer and two hidden layers.

In [13]:
tf.random.set_seed(0)
inputlayer = InputPCNLayer('InputLayer', units=5)
tf.random.set_seed(1)
dense1 = DensePCNLayer('HiddenLayer1', units=4, prev_layer_or_block=inputlayer, gamma_lr=1000.0)
tf.random.set_seed(2)
dense2 = DensePCNLayer('HiddenLayer2', units=3, prev_layer_or_block=dense1, gamma_lr=1000.0)
inputlayer.set_next_layer(dense1)
dense1.set_next_layer(dense2)
tf.random.set_seed(3)
test_x = tf.random.normal(shape=(2, 5))
test_net_act = dense2(dense1(inputlayer(test_x)))
print('Your net_act after the forward pass is:')
print(test_net_act.numpy(), 'and it should be:')
print('''[[-0.001 -0.001  0.001]
 [ 0.    -0.    -0.   ]]''')

print('dense1s state is:')
print(dense1.get_state().numpy(), 'and it should be:')
print('''[[ 0.017 -0.034 -0.027  0.006]
 [-0.026  0.005  0.018  0.016]]''')
print('Updating dense1s state using bottom-up and top-down signals!')
dense1.update_state()
print('dense1s new state based on the BU and TD pred errors is:')
print(dense1.get_state().numpy(), 'and it should be:')
print('''[[ 0.025 -0.051 -0.021  0.019]
 [-0.024  0.012  0.007  0.013]]''')

print('What the state of the input layer currently looks like:')
print(inputlayer.get_state().numpy())
print('Now lets update the state of the input layer while its clamped. It should not change vs what you see above.')
inputlayer.update_state()
print(inputlayer.get_state().numpy())
print('Now lets unclamp the input layer and update state again.')
inputlayer.unclamp_state()
inputlayer.update_state()
print('State after update:')
print(inputlayer.get_state().numpy(), 'It should change to:')
print('''[[-0.747 -1.644  0.113  1.059 -1.053]
 [-0.008  1.56  -0.931  0.124 -0.823]]''')

Your net_act after the forward pass is:
[[-0.001 -0.001  0.001]
 [ 0.    -0.    -0.   ]] and it should be:
[[-0.001 -0.001  0.001]
 [ 0.    -0.    -0.   ]]
dense1s state is:
[[ 0.017 -0.034 -0.027  0.006]
 [-0.026  0.005  0.018  0.016]] and it should be:
[[ 0.017 -0.034 -0.027  0.006]
 [-0.026  0.005  0.018  0.016]]
Updating dense1s state using bottom-up and top-down signals!
dense1s new state based on the BU and TD pred errors is:
[[ 0.025 -0.051 -0.021  0.019]
 [-0.024  0.012  0.007  0.013]] and it should be:
[[ 0.025 -0.051 -0.021  0.019]
 [-0.024  0.012  0.007  0.013]]
What the state of the input layer currently looks like:
[[-0.83  -1.827  0.125  1.177 -1.17 ]
 [-0.009  1.734 -1.034  0.138 -0.915]]
Now lets update the state of the input layer while its clamped. It should not change vs what you see above.
[[-0.83  -1.827  0.125  1.177 -1.17 ]
 [-0.009  1.734 -1.034  0.138 -0.915]]
Now lets unclamp the input layer and update state again.
State after update:
[[-0.747 -1.644  0.113  1

In [14]:
print('Now we can try to update dense2s state, but this should lead to a crash:')
print(dense2.update_state())

Now we can try to update dense2s state, but this should lead to a crash:


AttributeError: 'NoneType' object has no attribute 'predict_input'

### 1d. Questions

**Question 1:** Why does an attempt to update `dense2`'s state above (correctly) cause a crash?

**Answer 1:** Because dense2 is the top/output hidden layer, it does not have a next_layer above it. Its update_state() rule expects a top-down term from self.get_next_layer(), so when you try to update dense2, it crashes since that layer doesn’t exist.

### 1e. Implement and test `OutputPCNLayer`

This final type of `PCNLayer` is a `DensePCNLayer`, just adapted for the output layer. Because there are minimal differences compared to `DensePCNLayer`, the `OutputPCNLayer` class inherits from `DensePCNLayer` so conveniently you just need to implement the tiny differences! 😊

Similar to the input layer, the output layer behaves differently depending on when it is clamped.
- **If clamped:** We do not change the state.
- **If NOT clamped:** We do change the state because on the layer's BU input prediction error.

Recall that the output layer's state is updated according to gradient descent:

$$
\text{Mem}_{i, j} (t) = \text{Mem}_{i, j} (t-1) - \gamma \text{d}_{\text{BU}}
$$

where $\text{Mem}_{i, j} (t)$ is the medium term memory state of output layer neuron $j$ for sample $i$ at time step $t$, $\gamma$ is the feedback signal strength, and $\text{d}_{\text{BU}}$ is the "bottom-up gradient" (`shape: (B, C)`), which is the negated current layer prediction error "forwarded" through the current layer.

In [15]:
from dense_pcn_layer import OutputPCNLayer

#### Test `OutputPCNLayer`

A input → hidden → output PCN layer arrangement.

In [16]:
tf.random.set_seed(1)
inputlayer = InputPCNLayer('InputLayer', units=3)
tf.random.set_seed(2)
dense = DensePCNLayer('HiddenLayer', units=4, prev_layer_or_block=inputlayer, gamma_lr=2000.0)
tf.random.set_seed(3)
out = OutputPCNLayer('OutputLayer', units=5, prev_layer_or_block=dense, gamma_lr=2000.0)
inputlayer.set_next_layer(dense)
dense.set_next_layer(out)
tf.random.set_seed(4)
test_x = tf.random.normal(shape=(3, 3))
test_net_act = out(dense(inputlayer(test_x)))

print(f'Sanity check: the output layer is an output layer, right? {out.is_output_layer()}')
print('Output layer state currently after the full forward pass:')
print(out.get_state().numpy(), 'and it should be:')
print('''[[-0.001  0.     0.001 -0.001  0.   ]
 [-0.001  0.001  0.001 -0.001  0.   ]
 [ 0.    -0.    -0.     0.    -0.   ]]''')

print('Lets update the output layer state (while it is unclamped)!')
out.update_state()
print('Now the state is:')
print(out.get_state().numpy(), 'and it should be:')
print('''[[-1.271  0.744  1.445 -1.641  0.283]
 [-2.368  1.045  2.31  -1.752  0.22 ]
 [ 0.355 -0.272 -0.483  0.677 -0.129]]''')
print('Lets now clamp the output layer state and run another update state. The state should not change vs above.')
out.clamp_state()
out.update_state()
print('Now the state is:')
print(out.get_state().numpy())

Sanity check: the output layer is an output layer, right? True
Output layer state currently after the full forward pass:
[[-0.001  0.     0.001 -0.001  0.   ]
 [-0.001  0.001  0.001 -0.001  0.   ]
 [ 0.    -0.    -0.     0.    -0.   ]] and it should be:
[[-0.001  0.     0.001 -0.001  0.   ]
 [-0.001  0.001  0.001 -0.001  0.   ]
 [ 0.    -0.    -0.     0.    -0.   ]]
Lets update the output layer state (while it is unclamped)!
Now the state is:
[[-1.271  0.744  1.445 -1.641  0.283]
 [-2.368  1.045  2.31  -1.752  0.22 ]
 [ 0.355 -0.272 -0.483  0.677 -0.129]] and it should be:
[[-1.271  0.744  1.445 -1.641  0.283]
 [-2.368  1.045  2.31  -1.752  0.22 ]
 [ 0.355 -0.272 -0.483  0.677 -0.129]]
Lets now clamp the output layer state and run another update state. The state should not change vs above.
Now the state is:
[[-1.271  0.744  1.445 -1.641  0.283]
 [-2.368  1.045  2.31  -1.752  0.22 ]
 [ 0.355 -0.272 -0.483  0.677 -0.129]]


## Task 2: Implement the PC neural network

With all of the PCN layers implemented and tested, we are ready to assemble them into PCN networks with any number of hidden layers.

In [17]:
from dense_pcn import DensePCN

### 2a. Creating a Dense PCN network and performing the forward pass

Implement then test the following methods of the PC neural network class (`DensePCN`):
- constructor: creates and connects the layers together.
- `set_test_num_steps`: Sets the number of state updates to perform during inference / test time evaluation.
- `__call__`: Perform the forward pass thru the network.


#### Test Dense PCN network creation

In [18]:
test_net = DensePCN(input_feats_shape=(3,), C=5, hidden_units=(2, 4, 8, 16, 32), gamma_lr=500.0)
for layer in test_net.layers:
    print(layer)
print('Your network should look like:')
print('''InputLayer (3 units). Output shape: None
PredLayer_0 (2 units). Output shape: None
PredLayer_1 (4 units). Output shape: None
PredLayer_2 (8 units). Output shape: None
PredLayer_3 (16 units). Output shape: None
PredLayer_4 (32 units). Output shape: None
OutputLayer (5 units). Output shape: None''')

print('Here is the next layer of each layer:')
for layer in test_net.layers:
    next_layer = layer.get_next_layer()
    next_layer_str = next_layer.get_name() if next_layer is not None else '[None]'
    print(next_layer_str, end=' ')
print('you should see:')
print('PredLayer_0 PredLayer_1 PredLayer_2 PredLayer_3 PredLayer_4 OutputLayer [None] ')

print('Here is the prev layer of each layer:')
for layer in test_net.layers:
    next_layer = layer.get_prev_layer()
    next_layer_str = next_layer.get_name() if next_layer is not None else '[None]'
    print(next_layer_str, end=' ')
print('you should see:')
print('[None] InputLayer PredLayer_0 PredLayer_1 PredLayer_2 PredLayer_3 PredLayer_4')

print('Is each layer an output layer?')
for layer in test_net.layers:
    is_output = layer.is_output_layer()
    print(is_output, end=' ')
print('you should see:')
print('False False False False False False True')

InputLayer (3 units). Output shape: None
PredLayer_0 (2 units). Output shape: None
PredLayer_1 (4 units). Output shape: None
PredLayer_2 (8 units). Output shape: None
PredLayer_3 (16 units). Output shape: None
PredLayer_4 (32 units). Output shape: None
OutputLayer (5 units). Output shape: None
Your network should look like:
InputLayer (3 units). Output shape: None
PredLayer_0 (2 units). Output shape: None
PredLayer_1 (4 units). Output shape: None
PredLayer_2 (8 units). Output shape: None
PredLayer_3 (16 units). Output shape: None
PredLayer_4 (32 units). Output shape: None
OutputLayer (5 units). Output shape: None
Here is the next layer of each layer:
PredLayer_0 PredLayer_1 PredLayer_2 PredLayer_3 PredLayer_4 OutputLayer [None] you should see:
PredLayer_0 PredLayer_1 PredLayer_2 PredLayer_3 PredLayer_4 OutputLayer [None] 
Here is the prev layer of each layer:
[None] InputLayer PredLayer_0 PredLayer_1 PredLayer_2 PredLayer_3 PredLayer_4 you should see:
[None] InputLayer PredLayer_0 Pred

#### Test Dense PCN network forward pass

In [19]:
tf.random.set_seed(0)
test_net = DensePCN(input_feats_shape=(3,), C=5, hidden_units=(32, 64), gamma_lr=500.0, wt_scale=0.1)
tf.random.set_seed(1)
test_x = tf.random.normal(shape=(4, 3))
out_act = test_net(test_x)
print('The net_act in the output layer after the forward pass is:')
print(out_act.numpy(), 'and it should be:')
print('''[[-0.071 -0.092  0.002  0.047  0.125]
 [-0.01   0.109 -0.083  0.063 -0.036]
 [ 0.001  0.026 -0.019  0.001 -0.023]
 [-0.017  0.064 -0.058  0.049 -0.011]]''')


The net_act in the output layer after the forward pass is:
[[-0.071 -0.092  0.002  0.047  0.125]
 [-0.01   0.109 -0.083  0.063 -0.036]
 [ 0.001  0.026 -0.019  0.001 -0.023]
 [-0.017  0.064 -0.058  0.049 -0.011]] and it should be:
[[-0.071 -0.092  0.002  0.047  0.125]
 [-0.01   0.109 -0.083  0.063 -0.036]
 [ 0.001  0.026 -0.019  0.001 -0.023]
 [-0.017  0.064 -0.058  0.049 -0.011]]


### 2b. Updating states across the network

The main method in the PCN is the `update_states` method that performs state updates in each layer of the network for some number of iterations (`num_steps`) for the current mini-batch of data (`x_batch`). The state updates in the network work differently depending on whether the network is in training mode.

- **In training mode:** We want to learn from the class labels so we clamp the output layer state to the mini-batch labels (*in one-hot format*).
- **NOT in training mode:** We are not training so we really do not have (or want to have) access to the class labels of the current mini-batch of data. In this case, we want the output layer's state to evolve to hopefully end up consistent with the correct classes. So we do NOT clamp the output layer.

In either case, we do not allow the network to modify/corrupt the current mini-batch of data (*stays the "ground truth / objective reality"*) so we CLAMP the input layer to the current mini-batch.

Implement and test `update_states` in `DensePCN`.

#### Test `update_states` 1/2 training mode

In [20]:
tf.random.set_seed(0)
test_net = DensePCN(input_feats_shape=(3,), C=5, hidden_units=(4, 6), gamma_lr=150.0)
tf.random.set_seed(1)
test_x = tf.random.normal(shape=(2, 3))
test_y = tf.constant([0, 4])
test_net(test_x)

print('Lets simulate network state updates during training. We pass in the labels.')
test_net.update_states(num_steps=2, x_batch=test_x, y_batch=test_y)
for layer in test_net.layers[1:]:
    print(layer.get_name())
    print(layer.get_state().numpy())
print('You should see:')
print('''PredLayer_0
[[-5.711 -4.731 -5.154  6.629]
 [ 1.613  5.767  3.057  2.443]]
PredLayer_1
[[-350.646 -139.785  -27.298  -36.097 -368.811  338.781]
 [ 144.746 -270.268 -193.226  149.558 -125.801  142.857]]
OutputLayer
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]]''')

Lets simulate network state updates during training. We pass in the labels.
PredLayer_0
[[-5.711 -4.731 -5.154  6.629]
 [ 1.613  5.767  3.057  2.443]]
PredLayer_1
[[-350.646 -139.785  -27.298  -36.097 -368.811  338.781]
 [ 144.746 -270.268 -193.226  149.558 -125.801  142.857]]
OutputLayer
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]]
You should see:
PredLayer_0
[[-5.711 -4.731 -5.154  6.629]
 [ 1.613  5.767  3.057  2.443]]
PredLayer_1
[[-350.646 -139.785  -27.298  -36.097 -368.811  338.781]
 [ 144.746 -270.268 -193.226  149.558 -125.801  142.857]]
OutputLayer
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]]


#### Test `update_states` 2/2 inference mode

In [21]:
tf.random.set_seed(2)
test_x = tf.random.normal(shape=(1, 3))

print('Lets simulate network state updates during inference / test mode. We do not pass in labels.')
test_net.update_states(num_steps=3, x_batch=test_x)
for layer in test_net.layers[1:]:
    print(layer.get_name())
    print(layer.get_state().numpy())
print('You should see:')
print('''PredLayer_0
[[ 11.362 -14.729 -11.69   20.675]]
PredLayer_1
[[ 222.586  119.252 -130.505   75.245 -148.89  -162.756]]
OutputLayer
[[ 760.285  440.764  109.162 -127.549 -269.836]]''')


Lets simulate network state updates during inference / test mode. We do not pass in labels.
PredLayer_0
[[ 11.361 -14.73  -11.689  20.669]]
PredLayer_1
[[ 222.571  119.23  -130.496   75.243 -148.883 -162.729]]
OutputLayer
[[ 760.175  440.755  109.155 -127.484 -269.862]]
You should see:
PredLayer_0
[[ 11.362 -14.729 -11.69   20.675]]
PredLayer_1
[[ 222.586  119.252 -130.505   75.245 -148.89  -162.756]]
OutputLayer
[[ 760.285  440.764  109.162 -127.549 -269.836]]


Test with many iterations/steps

In [22]:
tf.random.set_seed(0)
test_net = DensePCN(input_feats_shape=(3,), C=5, hidden_units=(4, 6), gamma_lr=2.0, wt_scale=0.1)
tf.random.set_seed(1)
test_x = tf.random.normal(shape=(2, 3))
test_y = tf.constant([0, 4])
test_net(test_x)

print('Lets simulate network state updates during training for 25 iterations/steps.')
test_net.update_states(num_steps=25, x_batch=test_x, y_batch=test_y)
for layer in test_net.layers[1:]:
    print(layer.get_name())
    print(layer.get_state().numpy())
print('You should see:')
print('''PredLayer_0
[[-0.004  0.482  0.097 -0.059]
 [-0.189  0.177  0.046  0.828]]
PredLayer_1
[[-0.902 -0.022  1.088 -0.864  1.193  0.976]
 [ 0.535  0.988 -0.247  0.202 -0.179 -0.723]]
OutputLayer
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]]''')

Lets simulate network state updates during training for 25 iterations/steps.
PredLayer_0
[[-0.004  0.482  0.097 -0.059]
 [-0.189  0.177  0.046  0.828]]
PredLayer_1
[[-0.902 -0.022  1.088 -0.864  1.193  0.976]
 [ 0.535  0.988 -0.247  0.202 -0.179 -0.723]]
OutputLayer
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]]
You should see:
PredLayer_0
[[-0.004  0.482  0.097 -0.059]
 [-0.189  0.177  0.046  0.828]]
PredLayer_1
[[-0.902 -0.022  1.088 -0.864  1.193  0.976]
 [ 0.535  0.988 -0.247  0.202 -0.179 -0.723]]
OutputLayer
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]]


### 2c. PCN predictive loss

The goal of the predictive coding network is to minimize the **total energy** — i.e. the *squared* difference between the predicted and actual bottom-up input in each NON-INPUT layer of the net (*not just the output layer!!*). We implement this concept as the **predictive loss**: in each NON-INPUT layer we compute the sum-of-squares of the prediction errors across all the neurons. We average each layer's sum-of-squared prediction errors across data samples in the mini-batch to get the per-layer predictive loss Finally, we compute the total predictive loss as the sum of losses across layers.

In math:

$$
L
=
\sum_{l=1}^{L}
\frac{1}{N}
\sum_{i=1}^{N}
\sum_{j}
\frac{1}{2}
\varepsilon_{l,i,j}^{2}
$$

where $\varepsilon_{l,i,j}$ is the prediction error of layer $l$ formed to sample $i$ by neuron $j$.

Implement this predictive loss in the `loss` method of `DensePCN`.

*Note: We are not extending the `loss` method of DeepNetwork here. This is because the loss computation depends on the (cached) state in each layer. Interestingly, this means we do not need to pass the output layer netActs or the labels to compute the loss!*

#### Test: Predictive loss

In [23]:
tf.random.set_seed(0)
test_net = DensePCN(input_feats_shape=(3,), C=5, hidden_units=(4, 6), gamma_lr=2.5)
test_net.compile(loss='predictive')
tf.random.set_seed(1)
test_x = tf.random.normal(shape=(2, 3))
test_y = tf.constant([0, 4])
test_net(test_x)

print(f'Your initial loss is: {test_net.loss().numpy():.3f} and it should be 1.746')

print('Lets simulate network state updates during 1 training step.')
test_net.update_states(num_steps=10, x_batch=test_x, y_batch=test_y)
print(f'Your updated loss is: {test_net.loss().numpy():.3f} and it should be 3.217')

---------------------------------------------------------------------------
OutputLayer (5 units). Output shape: [1, 5]
PredLayer_1 (6 units). Output shape: [1, 6]
PredLayer_0 (4 units). Output shape: [1, 4]
InputLayer (3 units). Output shape: [1, 3]
---------------------------------------------------------------------------
Your initial loss is: 1.746 and it should be 1.746
Lets simulate network state updates during 1 training step.
Your updated loss is: 3.217 and it should be 3.217


### 2d. Train and test steps

Adapt your `DeepNetwork` train and test step code for corresponding methods in `DensePCN`. The main difference is your `update_states` method now replaces doing a normal forward pass through the network. Use the below code to test out your implementation.

In [24]:
tf.random.set_seed(0)
test_net = DensePCN(input_feats_shape=(5,), C=10, hidden_units=(5, 7), gamma_lr=1.0, train_num_steps=3)
test_net.compile(loss='predictive')
tf.random.set_seed(1)
test_x = tf.random.normal(shape=(3, 5))
test_y = tf.constant([9, 0, 4])

print('Your training loss is:')
tr_loss1 = test_net.train_step(test_x, test_y).numpy()
print(f'{tr_loss1:.3f}')
print('and it should be:\n1.505')

tf.random.set_seed(2)
test_x = tf.random.normal(shape=(3, 5))
test_y = tf.constant([1, 3, 8])

print('Your test acc/loss is:')
tr_acc, tr_loss2 = test_net.test_step(test_x, test_y, num_steps=2)
print(f'{tr_acc:.4f}/{tr_loss2:.4f}')
print('and it should be:\n0.0000/1.6901')

---------------------------------------------------------------------------
OutputLayer (10 units). Output shape: [1, 10]
PredLayer_1 (7 units). Output shape: [1, 7]
PredLayer_0 (5 units). Output shape: [1, 5]
InputLayer (5 units). Output shape: [1, 5]
---------------------------------------------------------------------------
Your training loss is:
1.505
and it should be:
1.505
Your test acc/loss is:
0.0000/1.6901
and it should be:
0.0000/1.6901
